In [1]:
 import pandas as pd

In [2]:
import numpy as np

In [3]:
mongo_data = [

    {
        "user_id": 101,
        "name": "Tanir",
        "age": 24,

        "preferences": {
            "city": "Kolkata",
            "budget": 15000,
            "preferred_bhk": 2
        },

        "visited_properties": [
            {
                "property_id": "P1",
                "rent": 12000,
                "area_sqft": 850
            },

            {
                "property_id": "P2",
                "rent": 18000,
                "area_sqft": 1200
            }
        ]
    },

    {
        "user_id": 102,
        "name": "Rahul",
        "age": 27,

        "preferences": {
            "city": "Bangalore",
            "budget": 25000,
            "preferred_bhk": 3
        },

        "visited_properties": [
            {
                "property_id": "P3",
                "rent": 22000,
                "area_sqft": 1400
            }
        ]
    }
]

In [4]:
df = pd.DataFrame(mongo_data)

In [5]:
df

,user_id,name,age,preferences,visited_properties
0,101,Tanir,24,"{'city': 'Kolkata', 'budget': 15000, 'preferre...","[{'property_id': 'P1', 'rent': 12000, 'area_sq..."
1,102,Rahul,27,"{'city': 'Bangalore', 'budget': 25000, 'prefer...","[{'property_id': 'P3', 'rent': 22000, 'area_sq..."


In [6]:
flat_df = pd.json_normalize(mongo_data)

In [7]:
flat_df

,user_id,name,age,visited_properties,preferences.city,preferences.budget,preferences.preferred_bhk
0,101,Tanir,24,"[{'property_id': 'P1', 'rent': 12000, 'area_sq...",Kolkata,15000,2
1,102,Rahul,27,"[{'property_id': 'P3', 'rent': 22000, 'area_sq...",Bangalore,25000,3


In [8]:
exploded_df = flat_df.explode("visited_properties")

In [9]:
exploded_df

,user_id,name,age,visited_properties,preferences.city,preferences.budget,preferences.preferred_bhk
0,101,Tanir,24,"{'property_id': 'P1', 'rent': 12000, 'area_sqf...",Kolkata,15000,2
0,101,Tanir,24,"{'property_id': 'P2', 'rent': 18000, 'area_sqf...",Kolkata,15000,2
1,102,Rahul,27,"{'property_id': 'P3', 'rent': 22000, 'area_sqf...",Bangalore,25000,3


In [10]:
property_df = pd.json_normalize(
    exploded_df["visited_properties"]
)

In [11]:
property_df

,property_id,rent,area_sqft
0,P1,12000,850
0,P2,18000,1200
1,P3,22000,1400


In [14]:
final_df = exploded_df.drop(
    columns=["visited_properties"]
).reset_index(drop=True)

In [15]:
final_df

,user_id,name,age,preferences.city,preferences.budget,preferences.preferred_bhk
0,101,Tanir,24,Kolkata,15000,2
1,101,Tanir,24,Kolkata,15000,2
2,102,Rahul,27,Bangalore,25000,3


In [17]:
property_df = property_df.reset_index(drop=True)

In [18]:
final_df = pd.concat(
    [final_df, property_df],
    axis=1
)

In [19]:
final_df

,user_id,name,age,preferences.city,preferences.budget,preferences.preferred_bhk,property_id,rent,area_sqft
0,101,Tanir,24,Kolkata,15000,2,P1,12000,850
1,101,Tanir,24,Kolkata,15000,2,P2,18000,1200
2,102,Rahul,27,Bangalore,25000,3,P3,22000,1400


In [20]:
final_df["affordability_ratio"] = (
    final_df["preferences.budget"]
    / final_df["rent"]
)

In [21]:
final_df

,user_id,name,age,preferences.city,preferences.budget,preferences.preferred_bhk,property_id,rent,area_sqft,affordability_ratio
0,101,Tanir,24,Kolkata,15000,2,P1,12000,850,1.250000
1,101,Tanir,24,Kolkata,15000,2,P2,18000,1200,0.833333
2,102,Rahul,27,Bangalore,25000,3,P3,22000,1400,1.136364


In [22]:
final_df["city_encoded"] = (
    final_df["preferences.city"]
    .astype("category")
    .cat.codes
)

In [23]:
final_df

,user_id,name,age,preferences.city,preferences.budget,preferences.preferred_bhk,property_id,rent,area_sqft,affordability_ratio,city_encoded
0,101,Tanir,24,Kolkata,15000,2,P1,12000,850,1.250000,1
1,101,Tanir,24,Kolkata,15000,2,P2,18000,1200,0.833333,1
2,102,Rahul,27,Bangalore,25000,3,P3,22000,1400,1.136364,0
